In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("cbb_data.csv")

df.columns = df.columns.str.replace(r"\s+", " ", regex=True).str.strip()

df["CLOSING SPREAD"] = pd.to_numeric(df["CLOSING SPREAD"], errors="coerce")
df["Team W/L (1/0)"] = pd.to_numeric(df["Team W/L (1/0)"], errors="coerce")
df["Team Over/Under Proj Team Total (1/0)"] = pd.to_numeric(
    df["Team Over/Under Proj Team Total (1/0)"], errors="coerce"
)

# Favorite rows only (1 per game)
fav = df[df["CLOSING SPREAD"] < 0].copy()

games = (
    fav.groupby("GAME-ID", as_index=False)
       .agg(
           SPREAD=("CLOSING SPREAD", "first"),
           FAV_WIN=("Team W/L (1/0)", "first"),
           FAV_OVER=("Team Over/Under Proj Team Total (1/0)", "first")
       )
)

# Underdog win = favorite loss
games["UDOG_WIN"] = 1 - games["FAV_WIN"]

# Underdog win + opponent over
games["UDOG_WIN_OPP_OVER"] = np.where(
    (games["FAV_WIN"] == 0) & (games["FAV_OVER"] == 0),
    1,
    0
)

# Spread buckets
games["spread_bucket"] = pd.cut(
    games["SPREAD"],
    bins=[-np.inf, -9, -5, -1],
    labels=["-9 and above", "-5 to -8.5", "-1 to -4.5"],
    right=True
)

# Table components
total = games.groupby("spread_bucket").size()
udog_wins = games[games["UDOG_WIN"] == 1].groupby("spread_bucket").size()
joint = games[games["UDOG_WIN_OPP_OVER"] == 1].groupby("spread_bucket").size()

prob = joint / udog_wins

fair_price = prob.apply(lambda p: -100*p/(1-p) if p > 0.5 else 100*(1-p)/p)

result = pd.DataFrame({
    "Total": total,
    "Total Udog Wins": udog_wins,
    "Total Udog Wins + Opp Over": joint,
    "P(Udog win + opp over | udog win)": prob.round(4),
    "Fair American Price without anticorrelation boost": fair_price.round(0)
}).fillna(0)

for c in ["Total", "Total Udog Wins", "Total Udog Wins + Opp Over"]:
    result[c] = result[c].astype(int)

result = result.reindex(["-1 to -4.5", "-5 to -8.5", "-9 and above"])

print(result)

               Total  Total Udog Wins  Total Udog Wins + Opp Over  \
spread_bucket                                                       
-1 to -4.5      2251              948                         693   
-5 to -8.5      1575              420                         319   
-9 and above    1934              160                         134   

               P(Udog win + opp over | udog win)  \
spread_bucket                                      
-1 to -4.5                                0.7310   
-5 to -8.5                                0.7595   
-9 and above                              0.8375   

               Fair American Price without anticorrelation boost  
spread_bucket                                                     
-1 to -4.5                                                -272.0  
-5 to -8.5                                                -316.0  
-9 and above                                              -515.0  


In [3]:
# Favorite rows only (1 row per game)
fav = df[df["CLOSING SPREAD"] < 0].copy()

games = (
    fav.groupby("GAME-ID", as_index=False)
       .agg(
           SPREAD=("CLOSING SPREAD","first"),
           FAV_WIN=("Team W/L (1/0)","first"),
           FAV_OVER=("Team Over/Under Proj Team Total (1/0)","first")
       )
)

# Underdog win = favorite loss
games["UDOG_WIN"] = 1 - games["FAV_WIN"]

# Underdog win + opponent over
games["UDOG_WIN_OPP_OVER"] = ((games["FAV_WIN"] == 0) & (games["FAV_OVER"] == 0)).astype(int)

# Spread buckets
games["spread_bucket"] = pd.cut(
    games["SPREAD"],
    bins=[-np.inf,-12,-8,-4,-1],
    labels=["-12 and above","-8 to -11.5","-4 to -7.5","-1 to -3.5"],
    right=True
)

# Table components
total = games.groupby("spread_bucket").size()
udog_wins = games[games["UDOG_WIN"]==1].groupby("spread_bucket").size()
joint = games[games["UDOG_WIN_OPP_OVER"]==1].groupby("spread_bucket").size()

prob = joint / udog_wins
fair_price = prob.apply(lambda p: -100*p/(1-p) if p>0.5 else 100*(1-p)/p)

result = pd.DataFrame({
    "Total": total,
    "Total Udog Wins": udog_wins,
    "Total Udog Wins + Opp Over": joint,
    "P(Udog win + opp over | udog win)": prob.round(4),
    "Fair American Price without anticorrelation boost": fair_price.round(0)
}).fillna(0)

for c in ["Total","Total Udog Wins","Total Udog Wins + Opp Over"]:
    result[c] = result[c].astype(int)

result = result.reindex(["-1 to -3.5","-4 to -7.5","-8 to -11.5","-12 and above"])

print(result)

               Total  Total Udog Wins  Total Udog Wins + Opp Over  \
spread_bucket                                                       
-1 to -3.5      1762              763                         548   
-4 to -7.5      1749              539                         413   
-8 to -11.5      973              164                         135   
-12 and above   1276               62                          50   

               P(Udog win + opp over | udog win)  \
spread_bucket                                      
-1 to -3.5                                0.7182   
-4 to -7.5                                0.7662   
-8 to -11.5                               0.8232   
-12 and above                             0.8065   

               Fair American Price without anticorrelation boost  
spread_bucket                                                     
-1 to -3.5                                                -255.0  
-4 to -7.5                                                -328.0  
-8 t

In [4]:
# Favorite rows only (1 row per game)
fav = df[df["CLOSING SPREAD"] < 0].copy()

games = (
    fav.groupby("GAME-ID", as_index=False)
       .agg(
           TOTAL=("CLOSING TOTAL", "first"),
           FAV_WIN=("Team W/L (1/0)", "first"),
           FAV_OVER=("Team Over/Under Proj Team Total (1/0)", "first")
       )
)

# Underdog win = favorite loss
games["UDOG_WIN"] = 1 - games["FAV_WIN"]

# Underdog win + opponent over
games["UDOG_WIN_OPP_OVER"] = ((games["FAV_WIN"] == 0) & (games["FAV_OVER"] == 0)).astype(int)

# Total buckets
games["total_bucket"] = pd.cut(
    games["TOTAL"],
    bins=[-np.inf, 144.5, 157.5, np.inf],
    labels=["144.5 and below", "145 to 157.5", "158 and above"],
    right=True
)

# Table components
total = games.groupby("total_bucket").size()
udog_wins = games[games["UDOG_WIN"] == 1].groupby("total_bucket").size()
joint = games[games["UDOG_WIN_OPP_OVER"] == 1].groupby("total_bucket").size()

prob = joint / udog_wins
fair_price = prob.apply(lambda p: -100*p/(1-p) if p > 0.5 else 100*(1-p)/p)

result = pd.DataFrame({
    "Total": total,
    "Total Udog Wins": udog_wins,
    "Total Udog Wins + Opp Over": joint,
    "P(Udog win + opp over | udog win)": prob.round(4),
    "Fair American Price without anticorrelation boost": fair_price.round(0)
}).fillna(0)

for c in ["Total", "Total Udog Wins", "Total Udog Wins + Opp Over"]:
    result[c] = result[c].astype(int)

result = result.reindex(["144.5 and below", "145 to 157.5", "158 and above"])

print(result)

                 Total  Total Udog Wins  Total Udog Wins + Opp Over  \
total_bucket                                                          
144.5 and below   2913              823                         619   
145 to 157.5      2413              620                         467   
158 and above      434               85                          60   

                 P(Udog win + opp over | udog win)  \
total_bucket                                         
144.5 and below                             0.7521   
145 to 157.5                                0.7532   
158 and above                               0.7059   

                 Fair American Price without anticorrelation boost  
total_bucket                                                        
144.5 and below                                             -303.0  
145 to 157.5                                                -305.0  
158 and above                                               -240.0  


In [9]:
# Favorite rows only (1 row per game)
fav = df[df["CLOSING SPREAD"] < 0].copy()

games = (
    fav.groupby("GAME-ID", as_index=False)
       .agg(
           TOTAL=("CLOSING TOTAL","first"),
           FAV_WIN=("Team W/L (1/0)","first"),
           FAV_OVER=("Team Over/Under Proj Team Total (1/0)","first")
       )
)

# Underdog win = favorite loss
games["UDOG_WIN"] = 1 - games["FAV_WIN"]

# Underdog win + opponent over
games["UDOG_WIN_OPP_OVER"] = ((games["FAV_WIN"] == 0) & (games["FAV_OVER"] == 0)).astype(int)

# Total buckets
games["total_bucket"] = pd.cut(
    games["TOTAL"],
    bins=[-np.inf,148.5,153.5,np.inf],
    labels=["148.5 and below","149 to 153.5","153.5 and above"],
    right=True
)

# Table components
total = games.groupby("total_bucket").size()
udog_wins = games[games["UDOG_WIN"]==1].groupby("total_bucket").size()
joint = games[games["UDOG_WIN_OPP_OVER"]==1].groupby("total_bucket").size()

prob = joint / udog_wins
fair_price = prob.apply(lambda p: -100*p/(1-p) if p>0.5 else 100*(1-p)/p)

result = pd.DataFrame({
    "Total": total,
    "Total Udog Wins": udog_wins,
    "Total Udog Wins + Opp Over": joint,
    "P(Udog win + opp over | udog win)": prob.round(4),
    "Fair American Price without anticorrelation boost": fair_price.round(0)
}).fillna(0)

for c in ["Total","Total Udog Wins","Total Udog Wins + Opp Over"]:
    result[c] = result[c].astype(int)

result = result.reindex(["148.5 and below","149 to 153.5","153.5 and above"])

print(result)

                 Total  Total Udog Wins  Total Udog Wins + Opp Over  \
total_bucket                                                          
148.5 and below   3939             1098                         807   
149 to 153.5       926              228                         185   
153.5 and above    895              202                         154   

                 P(Udog win + opp over | udog win)  \
total_bucket                                         
148.5 and below                             0.7350   
149 to 153.5                                0.8114   
153.5 and above                             0.7624   

                 Fair American Price without anticorrelation boost  
total_bucket                                                        
148.5 and below                                             -277.0  
149 to 153.5                                                -430.0  
153.5 and above                                             -321.0  
